# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Unit of Analysis + Time Window

**Unit of Analysis**

One row represents one content item for one client on one reporting date.

**Time Window**

I will use data from **March 2026 (2026-03)** because it is a mid-panel month that avoids using the final month as the development window. This follows the project guidance and reduces the risk of information leakage.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if token else "Token not found")
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    token=token
)

df_clients = dataset["train"].to_pandas()

print(df_clients.shape)
df_clients.head()

Token loaded successfully!
(104, 9)


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,None,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,None,None
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,None
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# Fields: Feature / Label / Context / Excluded

## Features
- impressions_90d
- ctr
- avg_position
- content_age_days
- days_since_last_update

These features are known before making a content refresh decision.

## Label / Proxy
Refresh priority score based on historical search performance.

## Context
- client_hash_id
- report_date
- content_id

These fields identify each observation but are not used as predictive features.

## Excluded
- trend_pct
- trend_direction
- is_declining_label

These fields are excluded because they are derived from the outcome and would cause data leakage.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df_clients.columns.tolist())
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git


['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Verification Queries

The verification queries confirm that the unit of analysis, time window, and selected fields match the assumptions described in the data contract.

The checks include:

- Grain verification
- Row count and date range
- Data availability and missing values

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

In [14]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{token}'
)
""")

In [15]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

The query confirms the number of records available for March 2026 in the daily performance table.

In [16]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {FACT}
""").df()





,rows
0,9841378


The returned minimum and maximum dates verify that only the selected month is included.

In [17]:
con.sql(f"""
SELECT
MIN(report_date),
MAX(report_date)
FROM {FACT}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min(report_date),max(report_date)
0,2026-03-01,2026-03-31


Filtering with `ga4_data_available IS TRUE` ensures only rows with valid GA4 metrics are used.

In [18]:
con.sql(f"""
SELECT COUNT(*) AS available_rows
FROM {FACT}
WHERE ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Data Limits

This dataset contains anonymized historical search and analytics information.

The data cannot prove that refreshing a page will improve search performance because search rankings depend on many external factors.

Some clients have limited historical data, and some rows may not contain complete analytics information. Therefore, the results should be interpreted as decision support rather than causal evidence.

In [19]:
con.sql(f"""
SELECT
COUNT(*) total_rows,
COUNT_IF(ga4_data_available IS TRUE) available_rows
FROM {FACT}
""").df()

,total_rows,available_rows
0,9841378,413966.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.